COCO class instance histogram (OWOD T1–T4)

In [ ]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

REPO = Path("..").resolve()
for p in (REPO, REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from common import resolve_path
from src.evaluation.owod_split import COCO_CLASSES, TASK_SPLITS

In [ ]:
COCO_JSONS = [
    REPO / "data/OWDETR/VOC2007/Annotations/instances_train2017.json",
    REPO / "data/OWDETR/VOC2007/Annotations/instances_val2017.json",
]

IMAGE_SPLIT_FILE = None
# IMAGE_SPLIT_FILE = REPO / "data/OWDETR/VOC2007/ImageSets/t1_train.txt"

OUT_DIR = REPO / "outputs/notebook/viz/coco_histogram_big"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TASK_COLORS = {
    1: "#1f77b4",
    2: "#ff7f0e",
    3: "#2ca02c",
    4: "#d62728",
}
TASK_LABELS = {t: f"T{t} ({len(TASK_SPLITS[t])} cls)" for t in TASK_SPLITS}

FIGSIZE_PER_TASK = (12, 10)
SAVE_DPI = 150

FONT_XTICK = 13         
FONT_YTICK = 13
FONT_AXIS_LABEL = 15     
FONT_TITLE = 15         
FONT_SUPTITLE = 15       
FONT_BAR_VALUE = 10      

In [ ]:
def load_image_ids(split_file):
    if split_file is None:
        return None
    path = resolve_path(split_file)
    ids = set()
    with open(path) as f:
        for line in f:
            stem = line.strip()
            if stem:
                ids.add(int(stem))
    return ids


def task_for_category(cat_id: int) -> int | None:
    for task, ids in TASK_SPLITS.items():
        if cat_id in ids:
            return task
    return None


def coco_instance_counts(json_paths, image_ids=None):
    counts = Counter()
    images_seen: dict[int, set[int]] = defaultdict(set)
    n_ann = 0
    n_skipped_img = 0

    for jp in json_paths:
        jp = resolve_path(jp)
        if not jp.exists():
            print(f"[warn] missing: {jp}")
            continue
        with open(jp) as f:
            data = json.load(f)

        if image_ids is not None:
            file_name_to_id = {}
            for im in data.get("images", []):
                stem = Path(im["file_name"]).stem
                file_name_to_id[stem] = im["id"]
            allowed = set()
            for iid in image_ids:
                stem = f"{iid:012d}"
                if stem in file_name_to_id:
                    allowed.add(file_name_to_id[stem])
        else:
            allowed = None

        for ann in data.get("annotations", []):
            if allowed is not None and ann["image_id"] not in allowed:
                n_skipped_img += 1
                continue
            cid = ann["category_id"]
            if cid not in COCO_CLASSES:
                continue
            counts[cid] += 1
            images_seen[cid].add(ann["image_id"])
            n_ann += 1

    meta = {
        "n_annotations": n_ann,
        "n_skipped_by_split": n_skipped_img,
        "image_split": str(IMAGE_SPLIT_FILE) if image_ids is not None else "all",
    }
    return counts, images_seen, meta


def counts_to_dataframe(instance_counts, images_per_class):
    rows = []
    for cat_id in sorted(COCO_CLASSES):
        task = task_for_category(cat_id)
        rows.append({
            "category_id": cat_id,
            "name": COCO_CLASSES[cat_id],
            "task": task,
            "instances": instance_counts.get(cat_id, 0),
            "images": len(images_per_class.get(cat_id, set())),
        })
    df = pd.DataFrame(rows)
    df["task_label"] = df["task"].map(TASK_LABELS)
    return df


def plot_per_task_panels(df, title_prefix, out_path=None):
    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE_PER_TASK, sharey=True)
    axes = axes.ravel()
    for ax, task in zip(axes, sorted(TASK_SPLITS)):
        sub = df[df["task"] == task]
        x = np.arange(len(sub))
        ax.bar(x, sub["instances"], color=TASK_COLORS[task])
        ax.set_xticks(x)
        ax.set_xticklabels(sub["name"], rotation=60, ha="right", fontsize=FONT_XTICK)
        ax.set_title(
            f"{TASK_LABELS[task]} — {sub['instances'].sum():,} instances",
            fontsize=FONT_TITLE,
        )
        ax.tick_params(axis="y", labelsize=FONT_YTICK)
        ax.set_ylabel("Instances", fontsize=FONT_AXIS_LABEL)
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle(title_prefix, y=1.02, fontsize=FONT_SUPTITLE)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=SAVE_DPI, bbox_inches="tight")
        print(f"Saved {out_path}")
    plt.show()


def plot_task_totals(df, title, out_path=None):
    totals = df.groupby("task", as_index=False)["instances"].sum()
    fig, ax = plt.subplots(figsize=(6, 4))
    tasks = totals["task"].tolist()
    ax.bar([TASK_LABELS[t] for t in tasks], totals["instances"],
           color=[TASK_COLORS[t] for t in tasks])
    ax.set_ylabel("Total instances", fontsize=FONT_AXIS_LABEL)
    ax.set_title(title, fontsize=FONT_TITLE)
    ax.tick_params(axis="both", labelsize=FONT_YTICK)
    ax.grid(axis="y", alpha=0.25)
    for i, v in enumerate(totals["instances"]):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=FONT_BAR_VALUE)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=SAVE_DPI, bbox_inches="tight")
        print(f"Saved {out_path}")
    plt.show()

In [ ]:
image_ids = load_image_ids(IMAGE_SPLIT_FILE)
if image_ids is not None:
    print(f"Image split: {len(image_ids):,} image stems from {IMAGE_SPLIT_FILE}")
else:
    print("Image split: all images in COCO JSONs")

instance_counts, images_per_class, meta = coco_instance_counts(COCO_JSONS, image_ids)
df = counts_to_dataframe(instance_counts, images_per_class)

print(f"Annotations counted: {meta['n_annotations']:,}")
display(df.groupby("task")[["instances", "images"]].sum().rename(columns={"images": "images_with_class"}))
display(df.sort_values(["task", "instances"], ascending=[True, False]).head(10))

In [ ]:
split_tag = "all" if image_ids is None else Path(IMAGE_SPLIT_FILE).stem
title = f"COCO instances per class ({split_tag})"

plot_per_task_panels(
    df, title,
    out_path=OUT_DIR / f"coco_instances_per_class_{split_tag}_by_task.pdf",
)
plot_task_totals(
    df, f"Total instances by OWOD task ({split_tag})",
    out_path=OUT_DIR / f"coco_instances_task_totals_{split_tag}.pdf",
)